# Signal Analysis Tool — Oscilloscope & PLECS

**Auto-detected formats:** LeCroy HDO/WS · Tektronix DPO · Siglent/Owon · PLECS variable-step · Generic numeric CSV

All analysis functions live in `signal_analysis_lib.py` (same folder).
**Only Cell 1 needs to be edited** for a normal analysis run.

---

## 1 — Configuration  ← edit this cell

In [23]:
# =============================================================================
#  USER CONFIGURATION  — only this cell needs to be edited for normal use
# =============================================================================

# --- FILES -------------------------------------------------------------------
CSV_FILES = [
    #r".cache\data4.csv"
    r"C:\Users\eliott.sefarang\Downloads\ROCOF_2Hz_ps_50Hz_49Hz.csv",
    # r"/path/to/your/file2.csv",
]

# --- TIME WINDOW  (None = full signal) ----------------------------------------
TIME_START = None   # e.g.  0.01  (seconds)
TIME_END   = None   # e.g.  0.05

# --- CHANNEL SELECTION -------------------------------------------------------
# None = keep ALL columns.
# CHANNELS = ["CH1", "CH2"]   or   ["DC voltage", "DC current"]
CHANNELS = ["I_grid_side_0", "w"]

# --- PLOT GROUPING  (applies per file independently) -------------------------
#   None         -> one subplot per channel (default, no overlay)
#   "all"        -> all channels on ONE shared subplot
#   [["A","B"],["C"]] -> custom groups; each inner list = one subplot
PLOT_GROUPS = None

# --- TIME-DOMAIN PLOT SIZE ---------------------------------------------------
PLOT_HEIGHT_PER_CH = 220   # px per subplot row
PLOT_WIDTH         = None  # total figure width (px) — None = fill cell

# --- FFT SETTINGS ------------------------------------------------------------
FUNDAMENTAL_FREQ = None   # Hz.  None = auto-detect from dominant peak
                          # Must be set if the signal window contains < 1 period.
FFT_MAX_HARMONIC = 1500     # how many harmonics to compute and display (H1 … HN)
FFT_SHOW_DC      = True   # show DC component (H0) as the first bar
FFT_TOP_N        = 5      # number of top harmonics shown in the summary table

# Window function applied before FFT.
# "rectangular"  no window — zero leakage when window covers exact integer
#                periods (PLECS-compatible, recommended for harmonic analysis)
# "hann"         Hann window — better sidelobe rejection but adds ±0.25
#                copies at adjacent bins (f0 ± df); use only if you need
#                to analyse non-periodic or burst signals
FFT_WINDOW = "rectangular"   # "rectangular" | "hann"

# Max samples fed into the FFT after resampling (controls speed vs. resolution)
FFT_MAX_POINTS = 500_000

# --- FFT PLOT SIZE -----------------------------------------------------------
FFT_HEIGHT_PER_CH = 280   # px per channel subplot
FFT_WIDTH         = None  # total figure width (px) — None = fill cell
FFT_SHOW_GRID     = True  # show gridlines on FFT plots

# --- POWER ANALYSIS ----------------------------------------------------------
POWER_PAIRS = [("3ph Meter:Measured voltage:1", "3ph Meter:Measured current:1")]   # list of (voltage_col, current_col) pairs — [] = skip

# --- DC RIPPLE ---------------------------------------------------------------
DC_CHANNELS = ["3ph Meter:Measured current:1"]   # channels that get DC ripple analysis

# --- PLOT LABEL CUSTOMISATION ------------------------------------------------
PLOT_TITLE = ""    # time-domain plot title ("" = auto)
FFT_TITLE  = ""    # FFT plot title ("" = auto)
YAXIS_LABELS = {}  # {ch: "label"} — y-axis label overrides
LEGEND_NAMES = {}  # {ch: "name"}  — legend name overrides

# --- FFT OVERLAY PLOT  (Section 11) ------------------------------------------
FFT_OVERLAY        = ["3ph Meter:Measured voltage:1"]     # channels to compare — [] = print available names
FFT_OVERLAY_FMIN   = 9000   # Hz — left edge of zoom (None = all)
FFT_OVERLAY_FMAX   = 11000   # Hz — right edge of zoom (None = all)
FFT_OVERLAY_TITLE  = ""     # "" = auto
FFT_OVERLAY_HEIGHT = None   # px — figure height (None = auto)
FFT_OVERLAY_WIDTH  = None   # px — figure width  (None = fill cell)

# =============================================================================
print("Configuration loaded.")
print(f"  Files        : {len(CSV_FILES)}")
print(f"  Time window  : {TIME_START} s  to  {TIME_END} s")
print(f"  Plot groups  : {PLOT_GROUPS}")
print(f"  f0 (FFT)     : {FUNDAMENTAL_FREQ} Hz  (None = auto)")
print(f"  FFT window   : {FFT_WINDOW}")


Configuration loaded.
  Files        : 1
  Time window  : None s  to  None s
  Plot groups  : None
  f0 (FFT)     : None Hz  (None = auto)
  FFT window   : rectangular


## 2 — Imports

In [24]:
import os
import sys
import numpy as np
import pandas as pd
from src.signal_analysis_lib import (
    PALETTE,
    load_csv,
    plot_time_domain,
    compute_statistics,
    compute_power,
    run_fft_all,
    plot_fft,
    table_harmonics,
    plot_custom_overlay,
    plot_fft_overlay,
)

print("Imports OK — library loaded from signal_analysis_lib.py")


Imports OK — library loaded from signal_analysis_lib.py


## 3 — Load Data

In [25]:
all_data = {}   # { file_label: DataFrame }

for path in CSV_FILES:
    label = os.path.splitext(os.path.basename(path))[0]
    print(f"Loading: {path}")
    try:
        df = load_csv(path)
    except Exception as e:
        print(f"  ERROR: {e}\n")
        continue

    sig_cols = [c for c in df.columns if c != "Time"]
    if CHANNELS:
        sig_cols = [c for c in sig_cols if c in CHANNELS]
    df = df[["Time"] + sig_cols]

    t0 = TIME_START if TIME_START is not None else df["Time"].iloc[0]
    t1 = TIME_END   if TIME_END   is not None else df["Time"].iloc[-1]
    df = df[(df["Time"] >= t0) & (df["Time"] <= t1)].copy().reset_index(drop=True)

    print(f"  Channels  : {sig_cols}")
    print(f"  Samples   : {len(df):,}")
    print(f"  Time span : {df['Time'].iloc[0]:.6g} s → {df['Time'].iloc[-1]:.6g} s\n")
    all_data[label] = df

if not all_data:
    raise RuntimeError("No data loaded — check CSV_FILES in cell 1.")
print(f"Total files loaded: {len(all_data)}")


Loading: C:\Users\eliott.sefarang\Downloads\ROCOF_2Hz_ps_50Hz_49Hz.csv
  Format detected : GENERIC
  Channels  : ['I_grid_side_0', 'w']
  Samples   : 59,891
  Time span : 0 s → 1.19998 s

Total files loaded: 1


## 4 — Time-Domain Plot

> One interactive figure per file.  Within a file, `PLOT_GROUPS` controls subplot layout.


In [26]:
plot_time_domain(
    all_data,
    plot_groups    = PLOT_GROUPS,
    title_override = PLOT_TITLE,
    height_per_ch  = PLOT_HEIGHT_PER_CH,
    fig_width      = PLOT_WIDTH,
    legend_names   = LEGEND_NAMES,
    yaxis_labels   = YAXIS_LABELS,
)


## 5 — Statistical Analysis

In [27]:
stat_df = compute_statistics(all_data, dc_channels=DC_CHANNELS)


=== Statistical Analysis ===


RMS Mean (DC)  AC RMS     Max  \
File                   Channel                                           
ROCOF_2Hz_ps_50Hz_49Hz I_grid_side_0  9.7342    0.4248  9.7249  21.578   
                       w              310.67    310.66  2.6551  314.17   

                                          Min Peak-Peak Std dev  
File                   Channel                                   
ROCOF_2Hz_ps_50Hz_49Hz I_grid_side_0  -19.516    41.094  9.7249  
                       w               307.79    6.3789  2.6551

## 6 — Power Analysis

In [28]:
compute_power(all_data, POWER_PAIRS)


=== Power Analysis ===
  Skip ROCOF_2Hz_ps_50Hz_49Hz: ['3ph Meter:Measured voltage:1', '3ph Meter:Measured current:1'] not found


## 7 — FFT Analysis  (harmonic spectrum)

> Rectangular window over exact integer periods — zero bin leakage, PLECS-compatible.
> Set `FFT_WINDOW = "hann"` in cell 1 for Hann windowing.
>
> **If you see** `⚠ window too short`: either set `FUNDAMENTAL_FREQ` to the actual
> fundamental or provide a longer signal containing ≥ 1 full period.


In [29]:
fft_results = run_fft_all(
    all_data,
    fundamental_freq = FUNDAMENTAL_FREQ,
    max_harm         = FFT_MAX_HARMONIC,
    window           = FFT_WINDOW,
    max_points       = FFT_MAX_POINTS,
)


=== FFT Analysis ===

  ROCOF_2Hz_ps_50Hz_49Hz / I_grid_side_0
  Auto f0   : 0.8333 Hz
  Resampled : 59,999 pts   fs = 5e+04 Hz
  n_periods : 1   N_per = 59999   N_trim = 59,999
  Window    : rectangular
  df        : 0.833347 Hz  (target 0.833347 Hz)
  FFT done  : [0.04 s]   THD = N/A

  ROCOF_2Hz_ps_50Hz_49Hz / w
  Auto f0   : 0.8333 Hz
  Resampled : 59,999 pts   fs = 5e+04 Hz
  n_periods : 1   N_per = 59999   N_trim = 59,999
  Window    : rectangular
  df        : 0.833347 Hz  (target 0.833347 Hz)
  FFT done  : [0.04 s]   THD = 44 %

FFT done.


In [30]:
plot_fft(
    fft_results,
    max_harm       = FFT_MAX_HARMONIC,
    show_dc        = FFT_SHOW_DC,
    show_grid      = FFT_SHOW_GRID,
    height_per_ch  = FFT_HEIGHT_PER_CH,
    fig_width      = FFT_WIDTH,
    title_override = FFT_TITLE,
    legend_names   = LEGEND_NAMES,
)


## 8 — FFT Harmonic Summary Table

In [31]:
print(f"=== Top-{FFT_TOP_N} harmonics per channel ===")
table_harmonics(fft_results, top_n=FFT_TOP_N, max_harm=FFT_MAX_HARMONIC)


=== Top-5 harmonics per channel ===


f0 (Hz) #1 H1  (0.8333 Hz)  \
File                   Channel                                    
ROCOF_2Hz_ps_50Hz_49Hz I_grid_side_0  0.8333                  —   
                       w              0.8333              3.437   

                                     #2 H2  (1.667 Hz) #3 H3  (2.5 Hz)  \
File                   Channel                                           
ROCOF_2Hz_ps_50Hz_49Hz I_grid_side_0                 —               —   
                       w                         0.862          0.6074   

                                     #4 H4  (3.333 Hz) #5 H5  (4.167 Hz)  
File                   Channel                                            
ROCOF_2Hz_ps_50Hz_49Hz I_grid_side_0                 —                 —  
                       w                        0.5346             0.403

---
## 9 — Custom Time-Domain Overlay

Re-plot any selection of channels with full style control.  
Edit **only this cell** and re-run it independently.


In [32]:
# =============================================================================
#  CUSTOM OVERLAY PLOT  ← edit freely and re-run this cell alone
# =============================================================================

# Channels to overlay.  Leave [] to print available channel names.
# Each entry: "ChannelName"  or  ("file_label", "ChannelName")
OVERLAY = ["3ph Meter:Measured voltage:1", "3ph Meter:Measured current:1"]

# Plot title & axis labels
CUSTOM_TITLE  = "Custom overlay"
CUSTOM_YLABEL = "Amplitude"
CUSTOM_XLABEL = "Time (s)"

# Background: "white" | "#f5f5f5" | "#1e1e1e" | "#0d1117" …
BG_COLOR = "white"

# Curve colours — [] = use default palette
CURVE_COLORS = []

# Line style — "solid" | "dot" | "dash" | "longdash" | "dashdot"
LINE_WIDTH = 1.6
LINE_DASH  = "solid"

# Legend name overrides — {"CH1": "Grid voltage"}
CUSTOM_LEGEND = {}

# Figure size
CUSTOM_HEIGHT = 500    # px
CUSTOM_WIDTH  = None   # px (None = fill cell)

# =============================================================================
plot_custom_overlay(
    overlay      = OVERLAY,
    all_data     = all_data,
    title        = CUSTOM_TITLE,
    ylabel       = CUSTOM_YLABEL,
    xlabel       = CUSTOM_XLABEL,
    bg_color     = BG_COLOR,
    curve_colors = CURVE_COLORS,
    line_width   = LINE_WIDTH,
    line_dash    = LINE_DASH,
    custom_legend= CUSTOM_LEGEND,
    height       = CUSTOM_HEIGHT,
    width        = CUSTOM_WIDTH,
)


No matching channels found.


---
## 10 — FFT Overlay Plot

Compare harmonics of multiple channels side-by-side on one figure.  
Set `FFT_OVERLAY` (and optional zoom) in **Cell 1**, then re-run this cell.


In [33]:
plot_fft_overlay(
    fft_overlay           = FFT_OVERLAY,
    fft_results           = fft_results,
    max_harm              = FFT_MAX_HARMONIC,
    show_dc               = FFT_SHOW_DC,
    show_grid             = FFT_SHOW_GRID,
    fmin                  = FFT_OVERLAY_FMIN,
    fmax                  = FFT_OVERLAY_FMAX,
    title_override        = FFT_OVERLAY_TITLE,
    height                = FFT_OVERLAY_HEIGHT,
    width                 = FFT_OVERLAY_WIDTH,
    height_per_ch_fallback= FFT_HEIGHT_PER_CH,
)


  Channel not found: '3ph Meter:Measured voltage:1'
No matching channels found.
